# Core clean + xG Dataset - Merge

> **Múltiples ligas**&nbsp;&nbsp;◦&nbsp;&nbsp;**Temporadas:** 2014-15 → 2023-24&nbsp;&nbsp;◦&nbsp;&nbsp;**Fuentes:** football-data.co.uk + Understat

## Objetivos

- Aplicar mapping de equipos y ligas, normalizar fechas y temporadas.
- Construir `match_id` compatible y realizar left join.
- Auditar el merge y analizar el dataset resultante.
- Exportar `core_enriched.parquet` (10.660 × 35).

## Estructura del Notebook

| # | Sección | Objetivo |
|---|---|---|
| 0 | Entorno y configuración | Librerías, rutas y configuraciones |
| 1 | Carga de datasets | Core clean y xG raw, verificación de dimensiones |
| 2 | Normalización del dataset xG | Equipos, fechas, ligas y temporadas |
| 3 | Alineación de `match_id` | Generación y verificación de la clave de join |
| 4 | Left join | Merge por `match_id` incorporando `home_xg` y `away_xg` |
| 5 | Auditoría del merge | Cobertura, integridad y consistencia |
| 6 | Análisis del `core_enriched` | Calibración xG + ventaja del equipo local |
| 7 | Conclusiones | Estado final del dataset |
| 8 | Exportación | Guardado del dataset enriquecido en Parquet |


---
##

## 0) Entorno y configuración

Configuración de dependencias, rutas del proyecto y parámetros de referencia.

### 0.1 Imports y rutas

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import json
from IPython.display import display, Markdown

def _find_project_root(start: Path) -> Path:
    markers = {"config", "src", "data"}
    for parent in [start, *start.parents]:
        if any((parent / m).exists() for m in markers):
            return parent
    raise RuntimeError(
        f"Raíz del proyecto no encontrada desde {start}. "
        f"Asegúrate de ejecutar el notebook desde dentro del proyecto."
    )

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

CONFIG_ROOT = PROJECT_ROOT / "config"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
RAW_XG_ROOT = PROJECT_ROOT / "data" / "processed" / "xg"
CORE_CLEAN_PATH = PROCESSED_ROOT / "multi_league" / "core_multi_league_clean.parquet"
XG_RAW_PATH = RAW_XG_ROOT / "xg_validated.parquet"
ENRICHED_PATH = PROCESSED_ROOT / "enriched" / "core_enriched.parquet"
ENRICHED_SCHEMA_PATH = PROCESSED_ROOT / "enriched" / "core_enriched_schema.json"

# Importación de funciones propias
from src.analysis import normalize_team_name
from src.utils import (
    get_shape_check,
    get_nulls_dict,
    get_duplicates_count,
    get_negatives_dict,
    get_join_integrity_check,
)

### 0.2 Mapeos de referencia

In [ ]:
with open(CONFIG_ROOT / "league_mapping.json") as f:
    LEAGUE_MAP = json.load(f)

with open(CONFIG_ROOT / "team_mapping_xg.json") as f:
    TEAM_MAP = json.load(f)

print(f"League mapping: {len(LEAGUE_MAP)} ligas")
print(f"Team mapping:   {len(TEAM_MAP)} equipos")

---
##

## 1) Carga de datasets

Lectura del core clean y del dataset xG raw validado.

In [ ]:
df_core = pd.read_parquet(CORE_CLEAN_PATH)
df_xg = pd.read_parquet(XG_RAW_PATH)

print(f"Core clean: {len(df_core):,} partidos × {len(df_core.columns)} columnas")
print(f"xG raw:     {len(df_xg):,} partidos × {len(df_xg.columns)} columnas")

assert len(df_core) == 10_660, f"Core: esperados 10,660, obtenidos {len(df_core):,}"
assert len(df_xg) == 10_660, f"xG: esperados 10,660, obtenidos {len(df_xg):,}"
print("\n✓ Ambos datasets con 10,660 filas")

---
##

## 2) Normalización del dataset xG

Tres transformaciones para alinear el dataset xG con el formato del core antes de construir `match_id`.

### 2.1 Equipos

Aplicación del diccionario de nombres generado en `04_eda_xg`. Los equipos sin entrada en el mapping conservan su nombre original.

In [ ]:
print("── Normalización de equipos ──────────────────────────────────────\n")
df_xg["home_team"] = df_xg["home_team"].map(TEAM_MAP).fillna(df_xg["home_team"])
df_xg["away_team"] = df_xg["away_team"].map(TEAM_MAP).fillna(df_xg["away_team"])
examples = list(TEAM_MAP.items())[:3]
print("  Ejemplos de mapping:")

for k, v in examples:
    print(f"   - {k:<20}  →  {v:<20}")

still_unknown = set(df_xg["home_team"].unique()) - set(df_core["HomeTeam"].unique())

if still_unknown:
    print(f"\n  ⚠ Equipos sin correspondencia: {len(still_unknown)}")
    print(f"  Ejemplos: {sorted(still_unknown)[:3]}")
else:
    print("\n  ✓ Mapping completo — todos los equipos tienen correspondencia\n")

print("──────────────────────────────────────────────────────────────────")

### 2.2 Ligas

Traducción de nombres de liga del formato Understat al formato football-data usando `LEAGUE_MAP`.

In [ ]:
print("── Normalización de ligas ───────────────────────────────────────\n")

df_xg["league"] = df_xg["league"].map(LEAGUE_MAP)

for us, core in LEAGUE_MAP.items():
    n = (df_xg["league"] == core).sum()
    print(f"   {us:<22}  →   {core:<12}   {n:>5,} partidos")

print("\n──────────────────────────────────────────────────────────────────")

### 2.3 Temporadas

Construir `Season` con el mismo formato que el core (`2015`, `2016`, ...) usando el corte en julio.

In [ ]:
SEASON_MAP = {f"{y}{y+1}": str(2000 + y + 1) for y in range(14, 24)}
df_xg["season"] = df_xg["season"].map(SEASON_MAP)

seasons_xg   = sorted(df_xg["season"].unique())
seasons_core = sorted(df_core["Season"].unique())

assert seasons_xg == seasons_core, f"Temporadas no coinciden: {set(seasons_xg) ^ set(seasons_core)}"

print("── Normalización de temporadas ───────────────────────────────────\n")

print(f"   Antes:    {'1415 … 2324':<15}")
print(f"   Después:  {seasons_xg[0]} … {seasons_xg[-1]:<6}   ({len(seasons_xg)} temporadas transformadas)")

print("\n   ✓ Temporadas alineadas con el core dataset")
print("\n──────────────────────────────────────────────────────────────────")

---
##

## 3) Alineación de `match_id`

### 3.1 Construcción del identificador

Generación del `match_id` en el dataset xG con el mismo formato que el core: `{League}_{Season}_{HomeTeam}_{AwayTeam}`, con nombres normalizados.

In [ ]:
df_xg["match_id"] = (
    df_xg["league"]
    + "_" + df_xg["season"]
    + "_" + df_xg["home_team"].apply(normalize_team_name).str.replace(" ", "_")
    + "_" + df_xg["away_team"].apply(normalize_team_name).str.replace(" ", "_")
)

n_unique = df_xg["match_id"].nunique()

if n_unique == len(df_xg):
    print(f"✓ match_id: {n_unique:,} claves únicas")
else:
    dups = df_xg[df_xg["match_id"].duplicated(keep=False)]
    print(f"⚠ {len(df_xg) - n_unique} match_id duplicados")
    display(dups[["match_id", "date", "league", "home_team", "away_team"]].head(10))

print("\nEjemplos:")
for league in sorted(df_xg["league"].unique()):
    sample = df_xg[df_xg["league"] == league]["match_id"].iloc[0]
    print(f"  • {league}: {sample}")

### 3.2 Verificación cruzada

Comparación directa del set de `match_id` entre ambos datasets.

In [ ]:
ids_core = set(df_core["match_id"])
ids_xg   = set(df_xg["match_id"])

common       = ids_core & ids_xg
only_in_core = ids_core - ids_xg
only_in_xg   = ids_xg   - ids_core

pct = len(common) / len(ids_core) * 100

bar = "█" * int(pct // 2) + "░" * (50 - int(pct // 2))

print("Cobertura del merge")
print("─────────────────────────────────────────────────────────────")
print(f" {bar} {pct:.2f}%")
print("─────────────────────────────────────────────────────────────\n")

print(f"  {'match_id en core':<18} │ {len(ids_core):>7,}")
print(f"  {'match_id en xG':<18} │ {len(ids_xg):>7,}")
print(f"  {'coincidencias':<18} │ {len(common):>7,}")
print(f"  {'solo en core':<18} │ {len(only_in_core):>7}")
print(f"  {'solo en xG':<18} │ {len(only_in_xg):>7}")

---
##

## 4) Left join

Merge del core clean con las columnas `home_xg` y `away_xg` del dataset xG por `match_id`.

In [ ]:
xg_for_merge = df_xg[["match_id", "home_xg", "away_xg"]].copy()

# Convertir xG a float64 estándar
xg_for_merge["home_xg"] = xg_for_merge["home_xg"].astype("float64")
xg_for_merge["away_xg"] = xg_for_merge["away_xg"].astype("float64")

df_enriched = df_core.merge(xg_for_merge, on="match_id", how="left")

print("Resultado del merge")
print("────────────────────────────────────\n")
print(f"  {'Core dataset':<20} │ {df_core.shape[0]:>7,} × {df_core.shape[1]}")
print(f"  {'Dataset enriquecido':<20} │ {df_enriched.shape[0]:>7,} × {df_enriched.shape[1]}")

new_cols = sorted(set(df_enriched.columns) - set(df_core.columns))
print(f"\n  {'Columnas añadidas':<20} │ {', '.join(new_cols)}")

---
##

## 5) Auditoría del merge

Validaciones de integridad sobre el dataset enriquecido.

In [ ]:
print("── Auditoría del merge ─────────────────────────────────────────\n")
expected_cols = len(df_core.columns) + len(xg_for_merge.columns) - 1
shape_ok, rows, cols = get_shape_check(df_enriched, expected_rows=10_660, expected_cols=expected_cols)
nulls_dict = get_nulls_dict(df_enriched[["home_xg", "away_xg"]])
no_nulls = len(nulls_dict) == 0
dups = get_duplicates_count(df_enriched, key_col="match_id")
no_dups = dups == 0
neg_dict = get_negatives_dict(df_enriched[["home_xg", "away_xg"]])
neg = sum(neg_dict.values())
no_neg = neg == 0
max_xg = max(df_enriched["home_xg"].max(), df_enriched["away_xg"].max())
in_range = max_xg <= 10

print(f"  {'✓' if shape_ok else '✗'} {'Shape':<16}   {rows:,} × {cols} (esperado: 10,660 × {expected_cols})")
null_txt = f"home_xg={nulls_dict.get('home_xg', 0)}, away_xg={nulls_dict.get('away_xg', 0)}"
print(f"  {'✓' if no_nulls else '✗'} {'Nulos en xG':<16}   {null_txt}")
print(f"  {'✓' if no_dups else '✗'} {'Duplicados id':<16}   {dups}")
print(f"  {'✓' if no_neg else '✗'} {'xG negativos':<16}   {neg}")
print(f"  {'✓' if in_range else '⚠'} {'xG máximo':<16}   {max_xg:.2f} (umbral: 10)")

print("\n── Goles core vs enriched (cross-check) ────────────────────────\n")
cross_check = get_join_integrity_check(df_core, df_enriched, "League")
for league, stats in cross_check.items():
    match, g_enriched, g_core = stats
    print(f"  [{match}] {league:<15} enriched={g_enriched:>6,}  core={g_core:>6,}")
print("\n───────────────────────────────────────────────────────────────")

all_ok = shape_ok and no_nulls and no_dups and no_neg and in_range
if all_ok:
    print("\n✓ Merge exitoso")
else:
    print("\n✗ Hay problemas que requieren revisión")

### 5.1 Preview del dataset enriquecido

In [ ]:
display(df_enriched[["match_id", "League", "Season", "HomeTeam", "AwayTeam", "home_xg", "away_xg"]].head(5).style
    .format({"home_xg": "{:.2f}", "away_xg": "{:.2f}"})
    .hide(axis="index")
    .set_table_styles([
        {"selector": "th, td", "props": [("text-align", "center")]},
        {"selector": "td:first-child", "props": [("text-align", "left")]},
    ])
)

---
##

## 6) Análisis del dataset enriquecido

Correlación entre los valores `home_xg` / `away_xg` integrados desde `Understat` y los goles reales. Valida que la señal xG tiene poder predictivo antes de usarla como feature en Fase 6.

### 6.1 Calibración del xG

Comparación entre los Expected Goals (xG) y los goles reales con el objetivo de analizar hasta qué punto el xG refleja el rendimiento ofensivo observado en los partidos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

pairs = [
    ("home_xg", "FTHG", "Local",     "YlGn"),
    ("away_xg", "FTAG", "Visitante", "YlOrRd"),
]

for ax, (xg_col, goals_col, label, cmap) in zip(axes, pairs):
    x = df_enriched[xg_col]
    y = df_enriched[goals_col]
    r = x.corr(y)

    hb = ax.hexbin(x, y, gridsize=30, cmap=cmap, mincnt=1, vmax=300)
    fig.colorbar(hb, ax=ax, label="Partidos")

    m, b = np.polyfit(x, y, 1)
    xline = np.linspace(x.min(), x.max(), 100)
    ax.plot(xline, m * xline + b, color="black", linewidth=1.5)

    ax.set_xlabel(f"xG {label}")
    ax.set_ylabel(f"Goles {label}")
    ax.set_title(f"{label} — r = {r:.3f}")
    ax.grid(alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("xG vs Goles reales (por partido)", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

### 6.2 Distribución de resultados por liga

Distribución `H` / `D` / `A` por liga sobre el dataset enriquecido. Cuantifica el efecto de jugar en casa en cada competición.

In [ ]:
pct_by_league = (
    df_enriched.groupby("League")["FTR"]
    .value_counts(normalize=True)
    .unstack()
    .reindex(columns=["H", "D", "A"])
    * 100
)

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(pct_by_league))
w = 0.25
colors = ["#2d8a4e", "#868e96", "#c23b2e"]
labels = ["Local (H)", "Empate (D)", "Visitante (A)"]

for i, (col, color, label) in enumerate(zip(["H", "D", "A"], colors, labels)):
    bars = ax.bar(x + i * w, pct_by_league[col], w, color=color, label=label)
    ax.bar_label(bars, fmt="%.1f", padding=3, fontsize=7.5)

ax.set_xticks(x + w)
ax.set_xticklabels([l.capitalize() for l in pct_by_league.index], fontsize=10)
ax.set_ylabel("% partidos", labelpad=10)
ax.set_ylim(0, 60)
ax.set_title("Distribución de resultados por liga", pad=14, fontsize=12)
ax.legend(loc="upper right", framealpha=0.7)
ax.yaxis.grid(True, alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

---
##

## 7) Conclusiones

El dataset `core_enriched` integra los **10.660 partidos** del core clean con las métricas de Expected Goals de Understat, resultando en un dataset de **10.660 × 35 columnas**.

### 7.1 Normalización

Se aplicaron tres transformaciones al dataset xG para alinear su nomenclatura con el core:

| Aspecto | Transformación |
|---------|---------------|
| Equipos | Nombres mapeados vía `team_mapping_xg.json` |
| Ligas | Understat → football-data (`ENG-Premier League` → `premier`, etc.) |
| Temporadas | Formato interno → año de cierre (`1415` → `2015`, etc.) |

### 7.2 Resultado del merge

- **Cobertura:** 100 % — los 10.660 `match_id` del core tienen correspondencia exacta en el dataset xG.
- **Columnas incorporadas:** `home_xg` y `away_xg`.
- **Sin nulos, duplicados ni negativos** en las columnas añadidas.
- **Goles cruzados** — coincidencia exacta entre ambas fuentes en las tres ligas.

### 7.3 Análisis exploratorio

- **Calibración xG** — correlación moderada (≈0.6) entre xG y goles reales: buena señal predictiva, aunque con variabilidad inherente al fútbol.
- **Ventaja del equipo local** — patrón consistente: local ~45%, empate ~25%, visitante ~30%, con ligeras variaciones entre ligas.

---
##

## 8) Exportación

Guardado del dataset enriquecido en formato Parquet con esquema JSON de referencia.

In [ ]:
df_enriched.to_parquet(ENRICHED_PATH, index=False)

schema = {
    "num_rows": len(df_enriched),
    "num_columns": len(df_enriched.columns),
    "num_leagues": df_enriched["League"].nunique(),
    "leagues": sorted(df_enriched["League"].unique().tolist()),
    "columns": sorted(df_enriched.columns.tolist()),
    "dtypes": {col: str(df_enriched[col].dtype) for col in sorted(df_enriched.columns)},
    "matches_per_league": df_enriched.groupby("League").size().to_dict(),
    "xg_source": "understat (vía soccerdata)"
}

with open(ENRICHED_SCHEMA_PATH, "w") as f:
    json.dump(schema, f, indent=2, ensure_ascii=False)

print(f"Dataset enriched exportado:")
print(f"  {len(df_enriched):,} filas × {len(df_enriched.columns)} columnas\n")

print(f"Archivos guardados:")
print(f"  · Dataset → {ENRICHED_PATH.relative_to(PROJECT_ROOT)}")
print(f"  · Esquema → {ENRICHED_SCHEMA_PATH.relative_to(PROJECT_ROOT)}")

---
##